In [26]:
import pandas as pd 
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report,accuracy_score
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [27]:
df=pd.read_csv("insurance.csv")

In [28]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
44,59,77.0,1.60,50.00,True,Lucknow,private_job,Medium
50,55,66.7,1.88,25.23,True,Jaipur,private_job,Medium
41,64,59.8,1.63,3.87,False,Mumbai,retired,Medium
51,45,101.9,1.62,28.95,True,Jaipur,private_job,High
57,72,76.8,1.69,1.36,True,Jalandhar,retired,High


In [29]:
df_feat=df.copy()

In [30]:
# feature 1 
df_feat["bmi"] = df_feat["weight"]/df_feat["height"] ** 2

In [31]:
# # feature 2 : age group
# def age_group(age):
#     if age<25:
#         return "Young"
#     elif age < 45:
#         return "Adult"
#     elif age <60:
#         return "middle_aged"
#     return "senior"

In [32]:
# df_feat["age"]=df_feat["age"].apply(age_group)

In [33]:
# feature 3 Life Style Risk 
def life_style_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "heigh"
    elif row["smoker"] and row["bmi"] >27:
        return "medium"
    else:
        return "low"

In [34]:
df_feat["lifestyle_risk"] = df_feat.apply(life_style_risk,axis=1)

In [35]:
df_feat=df_feat.drop(columns=["weight","height","smoker","city"])

In [36]:
X=df_feat[["bmi","age","lifestyle_risk","income_lpa","occupation"]]

In [37]:
Y=df_feat[["insurance_premium_category"]]

In [38]:
categorical_features=["lifestyle_risk","occupation"]
numerical_features= ["bmi","income_lpa","age"]

In [39]:
# create column transformer for One hote encoding
preprocessor = ColumnTransformer(

transformers = [
("cat",OneHotEncoder(),categorical_features),
("num","passthrough",numerical_features)
]

    
)

In [40]:
# create a pipe line with preprocessing and randomforest classifiers 
pipeline = Pipeline(
steps=[

    ("preprocessor ",preprocessor),
    ("classifier",XGBClassifier(
n_estimators=200,
learning_rate=0.1,
max_depth=5,
subsample=0.8,
colsample_bytree=0.8,
reg_alpha=0.1,
reg_lambda=1
))
]   
)

In [41]:

le = LabelEncoder()
Y= le.fit_transform(Y['insurance_premium_category'].values.ravel())

In [42]:
X

,bmi,age,lifestyle_risk,income_lpa,occupation
0,49.227482,67,low,2.92000,retired
1,30.189017,36,low,34.28000,freelancer
2,21.118382,39,low,36.64000,freelancer
3,45.535900,22,heigh,3.34000,student
4,24.296875,69,low,3.94000,retired
...,...,...,...,...,...
95,21.420747,36,low,19.64000,business_owner
96,47.984483,26,low,34.01000,private_job
97,18.765432,52,low,44.86000,freelancer
98,30.521676,27,low,28.30000,business_owner


In [43]:
Y

array([0, 1, 1, 2, 0, 2, 2, 2, 2, 1, 2, 1, 0, 1, 2, 2, 2, 2, 0, 0, 2, 0,
       1, 2, 2, 1, 2, 2, 1, 0, 1, 2, 2, 2, 0, 2, 2, 2, 0, 0, 2, 2, 2, 2,
       2, 0, 0, 2, 2, 2, 2, 0, 2, 2, 0, 1, 2, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [44]:
X_train,X_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=1)
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor ', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [45]:
# predict and evaluate 
accuracy_score(y_test,pipeline.predict(X_test))

0.6

In [46]:
import pickle

In [47]:
# save the trained usin pickle 
pickle_model_path= "Model.pkl"
with open(pickle_model_path,"wb") as f:
    pickle.dump(pipeline,f)

In [53]:
df_feat["occupation"].unique()

array(['retired', 'freelancer', 'student', 'government_job',
       'business_owner', 'unemployed', 'private_job'], dtype=object)